In [1]:
import geopandas as gpd
import pandas as pd

BASE = "/home/jovyan/work/children15mc"
RAW = f"{BASE}/data/raw"
PROC = f"{BASE}/data/processed"

print("Configuration complete.")

Configuration complete.


In [2]:
#1. LSOA Boundary
lsoa = gpd.read_file(f"{PROC}/lsoa_london.gpkg")
print(f"LSOA Boundary: {len(lsoa)} rows")

LSOA Boundary: 4994 rows


In [3]:
#2. IDACI(independent variable)
idaci = pd.read_csv(f"{RAW}/idaci_2019.csv", encoding="latin1")
code_col = [c for c in idaci.columns if "LSOA code" in c][0]
score_col = [c for c in idaci.columns if "IDACI" in c and "Score" in c][0]
idaci = idaci[[code_col, score_col]].rename(
    columns={code_col: "lsoa11cd", score_col: "idaci_score"})
print(f"IDACI: {len(idaci)} rows")

IDACI: 32844 rows


In [4]:
#2b. Convert IDACI from 2011 to 2021 LSOA codes
import numpy as np

lookup = pd.read_csv(f"{RAW}/lsoa_2011_2021_lookup.csv", encoding="utf-8-sig")
lookup = lookup[["LSOA11CD", "CHGIND", "LSOA21CD"]].drop_duplicates()
lookup.columns = ["lsoa11cd", "chgind", "lsoa21cd"]

print("Change indicator distribution:")
print(lookup["chgind"].value_counts())

idaci_map = lookup.merge(idaci, on="lsoa11cd", how="left")

idaci_map.loc[idaci_map["chgind"] == "X", "idaci_score"] = np.nan

idaci_2021 = (idaci_map[idaci_map["chgind"] != "X"]
              .groupby("lsoa21cd")["idaci_score"]
              .mean()
              .reset_index())

print(f"\nIDACI mapped to {len(idaci_2021)} 2021 LSOAs")

Change indicator distribution:
chgind
U    33647
S     1900
M      239
X       10
Name: count, dtype: int64

IDACI mapped to 35666 2021 LSOAs


In [5]:
#3. Child population (aged 0–14)
pop = pd.read_csv(f"{RAW}/census_age_lsoa.csv")
code_c = [c for c in pop.columns if "geography code" in c.lower()][0]
child_cols = [c for c in pop.columns if any(x in c for x in
              ["Aged 4 years and under", "Aged 5 to 9", "Aged 10 to 14"])]
pop["child_pop"] = pop[child_cols].sum(axis=1)
pop = pop[[code_c, "child_pop"]].rename(columns={code_c: "lsoa21cd"})
print(f"Child population: Used {child_cols}")

Child population: Used ['Age: Aged 4 years and under', 'Age: Aged 5 to 9 years', 'Age: Aged 10 to 14 years']


In [6]:
#4. Population-weighted centroid
cent = gpd.read_file(f"{RAW}/pop_centroids.geojson").to_crs(27700)
cent = cent[["LSOA21CD", "geometry"]].rename(columns={"LSOA21CD": "lsoa21cd"})
cent["cent_x"] = cent.geometry.x
cent["cent_y"] = cent.geometry.y
print(f"Centroid: {len(cent)} points")

Centroid: 35672 points


In [7]:
#5. Merge
df = lsoa.merge(idaci_2021, on="lsoa21cd", how="left")
df = df.merge(pop, on="lsoa21cd", how="left")
df = df.merge(cent[["lsoa21cd", "cent_x", "cent_y"]], on="lsoa21cd", how="left")

In [8]:
#6. Missing item check
print("Missing IDACI:", df.idaci_score.isna().sum())
print("Missing child population:", df.child_pop.isna().sum())
print("Missing centroid:", df.cent_x.isna().sum())
print("Total before merge:", len(df))

Missing IDACI: 0
Missing child population: 0
Missing centroid: 0
Total before merge: 4994


In [9]:
#7. Remove LSOAs with missing IDACI
df = df.dropna(subset=["idaci_score", "cent_x"]).reset_index(drop=True)
print("After dropping missing values:", len(df), "LSOAs remaining for analysis")

After dropping missing values: 4994 LSOAs remaining for analysis


In [10]:
#8. Save
df.to_file(f"{PROC}/analysis_table.gpkg", driver="GPKG")
df[["lsoa21cd", "lad22nm", "idaci_score", "child_pop", "cent_x", "cent_y"]].head()

,lsoa21cd,lad22nm,idaci_score,child_pop,cent_x,cent_y
0,E01000011,Barking and Dagenham,0.189,363,544370.330453,184729.246669
1,E01000046,Barking and Dagenham,0.208,473,547131.315622,184087.817083
2,E01000051,Barking and Dagenham,0.322,307,544557.364887,183744.134431
3,E01000077,Barking and Dagenham,0.294,393,547495.734300,185543.218170
4,E01000083,Barking and Dagenham,0.249,516,547666.816465,186006.968675
